# Controlled factorial 2×2 — Dev-only Kaggle training

This notebook runs three paired seeds (`2026`, `2126`, `2226`) for the small and expanded weak-label pools. It does not mount, read, or evaluate ViLexNorm Test. Each arm trains to eight epochs without early stopping; Dev-selected horizon-3 and horizon-8 artifacts are exported.

In [ ]:
from pathlib import Path
import shutil, subprocess

REPO = Path('/kaggle/working/VisolexNorm')
SOURCE_REF = 'main'  # replace with the committed controlled-experiment revision
REPOSITORY_URL = 'https://github.com/AIVIETNAM-AIO-DinhBao/VisolexNorm.git'
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', SOURCE_REF, REPOSITORY_URL, str(REPO)], check=True)
SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
print('Frozen source revision:', SOURCE_COMMIT)

In [ ]:
%cd {REPO}
!pip install -q -r requirements-kaggle.txt
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
print('GPU:', torch.cuda.get_device_name(0))

## Private input dataset
Attach a private Kaggle Dataset containing only `checkpoints/model_a/` and the four processed files required below. Do **not** upload `vilexnorm_test.jsonl` or `outputs/evaluation/`.

In [ ]:
MOUNT = Path('/kaggle/input/visolexnorm-controlled-input')  # update slug if needed
WORK = Path('/kaggle/working/controlled_factorial')
ARCHIVE = MOUNT / 'controlled_training_input.zip'
DATA = WORK / 'input' if ARCHIVE.is_file() else MOUNT
if ARCHIVE.is_file():
    if DATA.exists():
        shutil.rmtree(DATA)
    shutil.unpack_archive(ARCHIVE, DATA)
required = [
    DATA / 'checkpoints/model_a/config.json',
    DATA / 'data/processed/vilexnorm_train.jsonl',
    DATA / 'data/processed/vilexnorm_dev.jsonl',
    DATA / 'data/processed/visolex_weak_labeled.jsonl',
    DATA / 'data/processed/visolex_weak_labeled_expanded.jsonl',
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, f'Missing input files: {missing}'
assert not (DATA / 'data/processed/vilexnorm_test.jsonl').exists(), 'Do not attach Test.'
assert not (DATA / 'outputs/evaluation').exists(), 'Do not attach historical evaluation outputs.'
print('Dev-only input inventory verified.')

In [ ]:
!python -m pytest tests/training/test_controlled_experiments.py -q
!python -m scripts.controlled_experiments freeze-protocol --repo-root {DATA} --config {REPO}/configs/controlled_factorial_config.json --cmax-config {REPO}/configs/c_max20_early_stopping_config.json --output {WORK}/protocol.json

In [ ]:
# Build both paired manifests for each frozen seed.
for seed in (2026, 2126, 2226):
    subprocess.run([
        'python', '-m', 'scripts.controlled_experiments', 'build-factorial',
        '--repo-root', str(DATA), '--config', str(REPO/'configs/controlled_factorial_config.json'),
        '--protocol', str(WORK/'protocol.json'), '--seed', str(seed),
        '--output-dir', str(WORK/'runs'),
    ], check=True)

## Smoke gate
Run one 200+200 smoke job. Full runs must use separate work directories and should not use `--smoke-test`.

In [ ]:
SMOKE = WORK / 'smoke_seed_2026_small'
!python -m scripts.controlled_experiments train-factorial --model-a-checkpoint {DATA}/checkpoints/model_a --data-dir {DATA}/data/processed --manifest {WORK}/runs/seed_2026/small/mixture_manifest.json --config {REPO}/configs/controlled_factorial_config.json --work-dir {SMOKE} --smoke-test
import json
smoke = json.loads((SMOKE/'smoke_test.json').read_text())
assert smoke['passed'] and smoke['composition'] == {'gold': 200, 'pseudo': 200}, smoke
assert smoke['test_inputs_loaded'] is False, smoke
smoke

## Full factorial trajectories
This executes six full GPU trajectories. If Kaggle interrupts a trajectory, rerun only that command with `--resume` and the same `--work-dir`; do not rebuild its manifest or start a new directory.

In [ ]:
for seed in (2026, 2126, 2226):
    for arm in ('small', 'expanded'):
        run_dir = WORK / 'runs' / f'seed_{seed}' / arm
        subprocess.run([
            'python', '-m', 'scripts.controlled_experiments', 'train-factorial',
            '--model-a-checkpoint', str(DATA/'checkpoints/model_a'),
            '--data-dir', str(DATA/'data/processed'),
            '--manifest', str(run_dir/'mixture_manifest.json'),
            '--config', str(REPO/'configs/controlled_factorial_config.json'),
            '--work-dir', str(run_dir),
        ], check=True)

In [ ]:
!python -m scripts.controlled_experiments summarize-factorial --input-root {WORK}/runs --output {WORK}/factorial_summary.json
summary = json.loads((WORK/'factorial_summary.json').read_text())
assert summary['seeds'] == 3 and summary['test_metrics_used'] is False
summary['effects']

In [ ]:
# Exclude resumable optimizer states and six large model copies from the factorial archive.
# The archive retains frozen manifests, per-horizon Dev predictions/metrics and selection metadata; any checkpoint can be reconstructed from the committed source, Model A and the manifest.
for transient in list((WORK/'runs').glob('seed_*/*/state')) + list((WORK/'runs').glob('seed_*/*/best')):
    shutil.rmtree(transient)
shutil.make_archive('/kaggle/working/controlled_factorial_artifacts', 'zip', WORK, 'runs')
shutil.copy2(WORK/'protocol.json', '/kaggle/working/controlled_factorial_protocol.json')
shutil.copy2(WORK/'factorial_summary.json', '/kaggle/working/controlled_factorial_summary.json')
print('Download controlled_factorial_artifacts.zip, controlled_factorial_protocol.json, and controlled_factorial_summary.json.')